In [5]:
heuristic = {
    'A': 12, 'B': 4, 'C': 7, 'D': 3,
    'E': 8, 'F': 2, 'H': 4, 'I': 9,
    'S': 13, 'G': 0
}

graph = { 'S': {'A': 3, 'B': 2},
          'A': {'C': 4, 'D': 1},
          'B': {'E': 3, 'F': 1},
          'E': {'H': 5, 'I': 2},
          'F': {'I': 2, 'G': 3},
          'C': {}, 'D': {}, 'H': {}, 'I': {}, 'G': {} }

def greedy_bfs(graph, start, goal):
    frontier = [(start, heuristic[start])]
    visited = set()
    came_from = {start: None}

    while frontier:
        frontier.sort(key=lambda x: x[1])
        current_node, _ = frontier.pop(0)

        if current_node in visited:
            continue

        print(current_node, end=" ")
        visited.add(current_node)

        if current_node == goal:
            path = []
            while current_node is not None:
                path.append(current_node)
                current_node = came_from[current_node]
            path.reverse()
            print(f"\nGoal found with GBFS. Path: {path}")
            return

        for neighbor in graph[current_node]:
            if neighbor not in visited:
                came_from[neighbor] = current_node
                frontier.append((neighbor, heuristic[neighbor]))

    print("\nGoal not found")

print("Greedy Best First Search (S to G):")
greedy_bfs(graph, 'S', 'G')

Greedy Best First Search (S to G):
S B F G 
Goal found with GBFS. Path: ['S', 'B', 'F', 'G']


In [6]:
heuristic = {
    'S': 5, 'A': 3, 'B': 4,
    'C': 2, 'D': 6, 'G': 0
}

graph = {
    'S': {'A': 1, 'C': 4},
    'A': {'B': 2, 'D': 9},
    'B': {'D': 5},
    'C': {'G': 10},
    'D': {},
    'G': {}
}

def a_star(graph, start, goal):
    frontier = [(start, 0 + heuristic[start])]
    visited = set()
    g_costs = {start: 0}
    came_from = {start: None}

    while frontier:
        frontier.sort(key=lambda x: x[1])
        current_node, current_f = frontier.pop(0)

        if current_node in visited:
            continue

        print(current_node, end=" ")
        visited.add(current_node)

        if current_node == goal:
            path = []
            while current_node is not None:
                path.append(current_node)
                current_node = came_from[current_node]
            path.reverse()
            print(f"\nGoal found with A*. Path: {path}")
            return

        for neighbor, cost in graph[current_node].items():
            new_g_cost = g_costs[current_node] + cost
            f_cost = new_g_cost + heuristic[neighbor]

            if neighbor not in g_costs or new_g_cost < g_costs[neighbor]:
                g_costs[neighbor] = new_g_cost
                came_from[neighbor] = current_node
                frontier.append((neighbor, f_cost))

    print("\nGoal not found")

print("A* Search (S to D):")
a_star(graph, 'S', 'D')

A* Search (S to D):
S A C B G D 
Goal found with A*. Path: ['S', 'A', 'B', 'D']


In [7]:
from queue import PriorityQueue

maze_heuristic = [
    [10, 9,  8,  7,  6,  5,  4,  3,  2,  1,  0],
    [11, -1, -1, -1, -1, -1, -1, -1, -1, -1, 1],
    [12, -1, 10, 9,  8,  7,  6,  5,  4,  -1, 2],
    [13, -1, 11, -1, -1, -1, -1, -1, 5,  -1, 3],
    [14, 13, 12, -1, 10, 9,  8,  7,  6,  -1, 4],
    [-1, -1, 13, -1, 11, -1, -1, -1, -1, -1, 5],
    [16, 15, 14, -1, 12, 11, 10, 9,  8,  7,  6],
]

ROWS = 7
COLS = 11
START = (6, 0)
GOAL = (0, 10)

blocked = set()
for r in range(ROWS):
    for c in range(COLS):
        if maze_heuristic[r][c] == -1:
            blocked.add((r, c))

class Node:
    def __init__(self, position, parent=None):
        self.position = position
        self.parent = parent
        self.g = 0
        self.h = 0
        self.f = 0

    def __lt__(self, other):
        return self.f < other.f

def get_heuristic(pos):
    val = maze_heuristic[pos[0]][pos[1]]
    return val if val >= 0 else 0

def get_neighbors(pos):
    neighbors = []
    for dr, dc in [(1,0),(-1,0),(0,1),(0,-1)]:
        nr, nc = pos[0]+dr, pos[1]+dc
        if 0 <= nr < ROWS and 0 <= nc < COLS and (nr, nc) not in blocked:
            neighbors.append((nr, nc))
    return neighbors

def reconstruct_path(node):
    path = []
    while node:
        path.append(node.position)
        node = node.parent
    return path[::-1]

def greedy_best_first(start, goal):
    start_node = Node(start)
    start_node.h = get_heuristic(start)
    start_node.f = start_node.h

    frontier = PriorityQueue()
    frontier.put(start_node)
    visited = set()
    visited.add(start)

    while not frontier.empty():
        current = frontier.get()

        if current.position == goal:
            path = reconstruct_path(current)
            print(f"Greedy BFS Path (A to B): {path}")
            print(f"Total steps: {len(path)-1}")
            return path

        for neighbor_pos in get_neighbors(current.position):
            if neighbor_pos not in visited:
                visited.add(neighbor_pos)
                neighbor_node = Node(neighbor_pos, current)
                neighbor_node.h = get_heuristic(neighbor_pos)
                neighbor_node.f = neighbor_node.h
                frontier.put(neighbor_node)

    print("No path found (Greedy BFS)")
    return None

def a_star_maze(start, goal):
    start_node = Node(start)
    start_node.g = 0
    start_node.h = get_heuristic(start)
    start_node.f = start_node.g + start_node.h

    frontier = PriorityQueue()
    frontier.put(start_node)
    visited = {}
    visited[start] = 0

    while not frontier.empty():
        current = frontier.get()

        if current.position == goal:
            path = reconstruct_path(current)
            print(f"A* Path (A to B): {path}")
            print(f"Total steps: {len(path)-1}")
            return path

        for neighbor_pos in get_neighbors(current.position):
            new_g = current.g + 1
            if neighbor_pos not in visited or new_g < visited[neighbor_pos]:
                visited[neighbor_pos] = new_g
                neighbor_node = Node(neighbor_pos, current)
                neighbor_node.g = new_g
                neighbor_node.h = get_heuristic(neighbor_pos)
                neighbor_node.f = neighbor_node.g + neighbor_node.h
                frontier.put(neighbor_node)

    print("No path found (A*)")
    return None

print("Greedy Best First Search")
greedy_best_first(START, GOAL)

print("\n--- A* Best First Search ---")
a_star_maze(START, GOAL)

Greedy Best First Search
Greedy BFS Path (A to B): [(6, 0), (6, 1), (6, 2), (5, 2), (4, 2), (3, 2), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (3, 8), (4, 8), (4, 7), (4, 6), (4, 5), (4, 4), (5, 4), (6, 4), (6, 5), (6, 6), (6, 7), (6, 8), (6, 9), (6, 10), (5, 10), (4, 10), (3, 10), (2, 10), (1, 10), (0, 10)]
Total steps: 32

--- A* Best First Search ---
A* Path (A to B): [(6, 0), (6, 1), (6, 2), (5, 2), (4, 2), (4, 1), (4, 0), (3, 0), (2, 0), (1, 0), (0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 9), (0, 10)]
Total steps: 20


[(6, 0),
 (6, 1),
 (6, 2),
 (5, 2),
 (4, 2),
 (4, 1),
 (4, 0),
 (3, 0),
 (2, 0),
 (1, 0),
 (0, 0),
 (0, 1),
 (0, 2),
 (0, 3),
 (0, 4),
 (0, 5),
 (0, 6),
 (0, 7),
 (0, 8),
 (0, 9),
 (0, 10)]